In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# Stage 1 — global576 LoRA baseline (official)

Matched baseline for Stage 3 concat576:

```
[576 Projection-B global tokens] + format / Question / Answer
```

**Trainable:** Qwen LoRA adapters only  
**Frozen:** CLIP ViT-L/336 and Projection-B  
**Excluded:** Projection-A, region tokens, boxes, RAM++, and Grounding DINO

This notebook uses the same decontaminated fine-tuning pool, weighted source sampler, prompt, LoRA settings, and optimizer-step budget as Stage 3. Helpers live in `stage3_train.py` and `training.py`.

## 0. Dependencies

In [ ]:
import os
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_TORCH", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import importlib
import subprocess
import sys


def _ok(mod, attr=None):
    try:
        m = importlib.import_module(mod)
        return True if attr is None else hasattr(m, attr)
    except Exception:
        return False


if not _ok("torch", "Tensor"):
    raise RuntimeError(
        "Broken PyTorch. Restore it with conda; do not pip-install torch here."
    )

to_install = []
if not _ok("peft"):
    to_install.append("peft>=0.14.0")
if not _ok("transformers"):
    to_install.append("transformers>=4.48.0")
if not _ok("accelerate"):
    to_install.append("accelerate>=0.34.0")

import numpy as np
if int(np.__version__.split(".")[0]) >= 2:
    to_install.append("numpy<2.0")

if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install, "-q"])
    print("Installed:", ", ".join(to_install))
    print("Restart the kernel, then continue from section 1.")
else:
    import torch, transformers, peft
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
    print("transformers", transformers.__version__, "peft", peft.__version__)

## 1. Paths and hyperparameters

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

PROJECT_DIR = Path(".").resolve()
CLEAN_PKL = Path(os.environ.get("REVA_STAGE3_CLEAN_PKL", Path(os.environ["REVA_DECONTAMINATION_ROOT"]) / "stage3_eval_clean.pkl"))
PROJECTION_B_WEIGHTS = PROJECT_DIR / "projection_b_best_weights.pt"

RUN_TRAINING = True
MAX_TRAIN_SAMPLES = None       # e.g. 2000 only for a smoke run
MAX_OPTIMIZER_STEPS = 10000   # match the completed Stage 3 run
GRAD_ACCUM = 16
PEAK_LR = 1e-4

TAG = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(os.environ["REVA_CHECKPOINT_ROOT"]) / f"global576_stage1_{TAG}"

assert CLEAN_PKL.exists(), CLEAN_PKL
assert PROJECTION_B_WEIGHTS.exists(), PROJECTION_B_WEIGHTS
print("Output:", OUTPUT_DIR)

## 3. Configure and load models

In [ ]:
from reva.config import Stage3Config
from reva.models import load_frozen_vit, load_projection_b_weights, load_qwen_with_lora

config = Stage3Config(
    clean_pkl_path=CLEAN_PKL,
    projection_b_path=str(PROJECTION_B_WEIGHTS),
    lora_output_dir=OUTPUT_DIR,
    training_log_path=OUTPUT_DIR / "training_log.json",
    max_optimizer_steps=MAX_OPTIMIZER_STEPS,
    max_train_samples=MAX_TRAIN_SAMPLES,
    gradient_accumulation_steps=GRAD_ACCUM,
    peak_learning_rate=PEAK_LR,
    dataloader_num_workers=0,
)

frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(str(PROJECTION_B_WEIGHTS), config)
qwen, qwen_tokenizer = load_qwen_with_lora(config)

qwen.config.use_cache = False
qwen.print_trainable_parameters()
print("device:", config.device, "| dtype:", config.compute_dtype)

## 4. Load clean pool (same pickle + weighted mix as Stage 3)

In [ ]:
from reva.stage3_train import load_stage3_clean_pool

train_samples = load_stage3_clean_pool(
    config.clean_pkl_path,
    max_samples=config.max_train_samples,
)

sample = train_samples[0]
print("Example source:", sample["source"])
print("Question:", sample["question"][:120])
print("Answer:", sample["answer"][:80])
print("Boxes in pickle are ignored:", len(sample.get("boxes") or []))

## 5. Train global-only LoRA

In [ ]:
from reva.training import train_stage3_lora

if RUN_TRAINING:
    qwen, training_log = train_stage3_lora(
        qwen=qwen,
        qwen_tokenizer=qwen_tokenizer,
        frozen_vit=frozen_vit,
        projection_head_b=projection_head_b,
        region_extractor=None,
        train_samples=train_samples,
        clip_image_processor=clip_image_processor,
        config=config,
        include_regions=False,
    )
    print("Best Stage 1 LoRA:", config.lora_output_dir / config.lora_best_subdir)
else:
    print("Skipping training")

## 6. Verify saved outputs

In [ ]:
best_lora = config.lora_output_dir / config.lora_best_subdir
final_lora = config.lora_output_dir / "final_lora"
manifest = config.lora_output_dir / "stage1_global_lora_training_manifest.json"
log_path = Path(config.training_log_path)

assert best_lora.exists(), best_lora
assert final_lora.exists(), final_lora
assert manifest.exists(), manifest
assert log_path.exists(), log_path

print("Best adapter:", best_lora)
print("Final adapter:", final_lora)
print("Manifest:", manifest)
print("Training log:", log_path)

In [ ]:
from peft import PeftModel

base_qwen = qwen.get_base_model() if hasattr(qwen, "get_base_model") else qwen
qwen_eval = PeftModel.from_pretrained(base_qwen, str(best_lora), is_trainable=False)
qwen_eval.eval()
frozen_vit.eval()
projection_head_b.eval()
print("Loaded", best_lora)

In [ ]:
import json
import random
from pathlib import Path

from tqdm.auto import tqdm
from reva.evaluation import download_gqa_val, download_vqav2_val, _vqav2_image_path
from reva.inference import normalise_answer, run_concat576_inference, vqa_soft_score
from reva.stage3_dataset import (
    DEFAULT_FORMAT_PROMPT,
    _iter_visual7w_qa_pairs,
    _resolve_visual7w_dataset_json,
    _visual7w_boxes_by_id,
)


def clean_pred(text):
    return text.strip().split("\n")[0].strip().rstrip(".")


def score(pred, sample):
    if "answers" in sample:
        return vqa_soft_score(pred, sample["answers"])
    return float(
        normalise_answer(pred) == normalise_answer(sample["answer"])
    )


def eval_stage1_bench(samples, n, desc):
    rng = random.Random(EVAL_SEED)
    pool = samples[:]
    rng.shuffle(pool)
    pool = pool[:min(n, len(pool))]

    total = 0.0

    for sample in tqdm(pool, desc=desc):
        pred, aux = run_concat576_inference(
            sample["image_path"],
            sample["question"],
            boxes_px=[],
            include_regions=False,
            frozen_vit=frozen_vit,
            projection_head_b=projection_head_b,
            region_extractor=None,
            qwen=qwen_eval,
            qwen_tokenizer=qwen_tokenizer,
            clip_image_processor=clip_image_processor,
            config=config,
            format_prompt=sample.get(
                "format_prompt",
                DEFAULT_FORMAT_PROMPT,
            ),
        )

        # Ensure no region tokens accidentally entered the model.
        assert aux["num_region_tokens"] == 0
        assert aux["num_boxes"] == 0

        total += score(clean_pred(pred), sample)

    n_eff = len(pool)

    return {
        "global576_pct": 100.0 * total / n_eff if n_eff else 0.0,
        "n": n_eff,
    }


def build_v7w(v7w_dir, split="val"):
    """
    Build the same Visual7W pointing subset used by the Stage 3 diagnostic.

    Box annotations are used only to recover the textual answer label.
    No boxes are passed to the Stage 1 model.
    """
    v7w_dir = Path(v7w_dir)
    dataset_json = _resolve_visual7w_dataset_json(v7w_dir)

    if dataset_json is None:
        return []

    with open(dataset_json) as f:
        data = json.load(f)

    boxes_by_id = _visual7w_boxes_by_id(data)
    samples = []

    for qa, image_meta in _iter_visual7w_qa_pairs(data, split=split):
        qa_type = str(
            qa.get("type", qa.get("qa_type", ""))
        ).lower()

        if qa_type and qa_type not in ("which", "where"):
            continue

        rel = image_meta.get(
            "filename",
            image_meta.get(
                "image_path",
                qa.get("image_path", ""),
            ),
        )

        if not rel:
            continue

        image_path = v7w_dir / rel

        if not image_path.exists():
            image_path = dataset_json.parent / rel

        if not image_path.exists():
            image_path = (
                v7w_dir
                / "visual7w_pointing/images"
                / Path(rel).name
            )

        if not image_path.exists():
            continue

        answer = str(
            qa.get(
                "multiple_choice_answer",
                qa.get("answer_text", qa.get("answer", "")),
            )
        )

        answer_box = None

        if qa.get("answer") is not None:
            try:
                answer_box = boxes_by_id.get(int(qa["answer"]))
            except (TypeError, ValueError):
                pass

        if answer_box is not None:
            answer = str(answer_box.get("name", answer))

        question = qa.get(
            "question",
            qa.get("qa_text", qa.get("question_text", "")),
        )

        if answer_box is not None and answer and question:
            samples.append({
                "image_path": str(image_path),
                "question": str(question),
                "answer": answer,
            })

    print(f"Visual7W candidates: {len(samples):,}")
    return samples


RUN_EVAL = True
EVAL_N = 2000
EVAL_SEED = 42
GQA_ROOT = Path(os.environ["REVA_GQA_ROOT"])
VQAV2_ROOT = Path(os.environ["REVA_VQAV2_ROOT"])
VISUAL7W_ROOT = Path(os.environ["REVA_VISUAL7W_ROOT"])

if RUN_EVAL:
    results = {}

    # GQA diagnostic
    gqa_samples, gqa_image_dir = download_gqa_val(GQA_ROOT)

    gqa_eval = [
        {
            "image_path": str(
                gqa_image_dir / f"{sample['image_id']}.jpg"
            ),
            "question": sample["question"],
            "answer": sample["answer"],
        }
        for sample in gqa_samples
    ]

    results["gqa"] = eval_stage1_bench(
        gqa_eval,
        EVAL_N,
        f"Stage 1 GQA n={EVAL_N}",
    )

    # VQAv2 diagnostic with soft accuracy
    vqa_samples, vqa_image_dir = download_vqav2_val(VQAV2_ROOT)

    vqa_eval = [
        {
            "image_path": str(
                _vqav2_image_path(
                    "val",
                    vqa_image_dir,
                    sample["image_id"],
                )
            ),
            "question": sample["question"],
            "answers": sample["answers"],
        }
        for sample in vqa_samples
    ]

    results["vqav2"] = eval_stage1_bench(
        vqa_eval,
        EVAL_N,
        f"Stage 1 VQAv2 n={EVAL_N}",
    )

    # Visual7W diagnostic
    visual7w_eval = build_v7w(VISUAL7W_ROOT)

    results["visual7w"] = eval_stage1_bench(
        visual7w_eval,
        min(EVAL_N, len(visual7w_eval)),
        "Stage 1 Visual7W",
    )

    print("\n=== Stage 1 global576 LoRA ===")

    for benchmark, result in results.items():
        print(
            f"{benchmark:10s} "
            f"global576={result['global576_pct']:.2f}%  "
            f"n={result['n']}"
        )

    payload = {
        "experiment": "stage1_global576_lora",
        "visual_prefix_tokens": 576,
        "regions_used": False,
        "results": results,
        "output_dir": str(config.lora_output_dir),
    }

    result_path = (
        config.lora_output_dir
        / f"stage1_global576_eval_n{EVAL_N}.json"
    )

    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2)

    print("Saved:", result_path)

else:
    print("Skipping evaluation")

In [ ]:
from reva.evaluation import load_stage1_eval_stack

stack1 = load_stage1_eval_stack(
    projection_b_path=Path(os.environ.get("REVA_PROJECTION_B_WEIGHTS", "projection_b_best_weights.pt")),
    lora_path=Path(os.environ.get("REVA_STAGE1_ADAPTER", "weights/lora_adapter_weights/stage1_lora")),
)

In [ ]:
from reva.inference import run_concat576_inference

IMAGES_ROOT = Path(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"))
FORMAT_PROMPT = "Answer in ten words and not in a single word."

def ask_stage1(
    image_path_or_url: str,
    question: str,
    *,
    max_new_tokens=32,
):
    pred, aux = run_concat576_inference(
        image_path_or_url,
        question,
        frozen_vit=stack1.frozen_vit,
        projection_head_b=stack1.projection_head_b,
        region_extractor=None,
        qwen=stack1.qwen,
        qwen_tokenizer=stack1.tokenizer,
        clip_image_processor=stack1.image_processor,
        config=stack1.config,
        boxes_px=[],
        format_prompt=FORMAT_PROMPT,
        max_new_tokens=max_new_tokens,
        include_regions=False,  # global576 only
    )
    answer = str(pred).strip().splitlines()[0].strip().rstrip(".")
    print("\nAnswer:", answer)
    print(f"Prefix: {aux.get('num_global_tokens', 576)} global + 0 region (0 boxes)")
    return {
        "answer": answer,
        "num_regions": 0,
        "num_global_tokens": aux.get("num_global_tokens", 576),
        "num_region_tokens": 0,
    }

while True:
    image_input = input("\n[Stage1] Image path/URL/filename (or quit): ").strip()
    if image_input.lower() in {"quit", "q", "exit"}:
        break

    if image_input.startswith("http") or Path(image_input).is_file():
        image_path = image_input
    else:
        image_path = str(IMAGES_ROOT / image_input)

    question = input("Question (or quit): ").strip()
    if question.lower() in {"quit", "q", "exit"}:
        break
    if not question:
        continue

    ask_stage1(image_path, question)

In [ ]:
from reva.inference import run_concat576_inference

IMAGES_ROOT = Path(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"))
FORMAT_PROMPT = "Answer in ten words and not in a single word."

def ask_stage1(
    image_path_or_url: str,
    question: str,
    *,
    max_new_tokens=32,
):
    pred, aux = run_concat576_inference(
        image_path_or_url,
        question,
        frozen_vit=stack1.frozen_vit,
        projection_head_b=stack1.projection_head_b,
        region_extractor=None,
        qwen=stack1.qwen,
        qwen_tokenizer=stack1.tokenizer,
        clip_image_processor=stack1.image_processor,
        config=stack1.config,
        boxes_px=[],
        format_prompt=FORMAT_PROMPT,
        max_new_tokens=max_new_tokens,
        include_regions=False,  # global576 only
    )
    answer = str(pred).strip().splitlines()[0].strip().rstrip(".")
    print("\nAnswer:", answer)
    print(f"Prefix: {aux.get('num_global_tokens', 576)} global + 0 region (0 boxes)")
    return {
        "answer": answer,
        "num_regions": 0,
        "num_global_tokens": aux.get("num_global_tokens", 576),
        "num_region_tokens": 0,
    }

while True:
    image_input = input("\n[Stage1] Image path/URL/filename (or quit): ").strip()
    if image_input.lower() in {"quit", "q", "exit"}:
        break

    if image_input.startswith("http") or Path(image_input).is_file():
        image_path = image_input
    else:
        image_path = str(IMAGES_ROOT / image_input)

    question = input("Question (or quit): ").strip()
    if question.lower() in {"quit", "q", "exit"}:
        break
    if not question:
        continue

    ask_stage1(image_path, question)

In [ ]:
from reva.inference import run_concat576_inference

IMAGES_ROOT = Path(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"))
FORMAT_PROMPT = "Answer in ten words and not in a single word."

def ask_stage1(
    image_path_or_url: str,
    question: str,
    *,
    max_new_tokens=32,
):
    pred, aux = run_concat576_inference(
        image_path_or_url,
        question,
        frozen_vit=stack1.frozen_vit,
        projection_head_b=stack1.projection_head_b,
        region_extractor=None,
        qwen=stack1.qwen,
        qwen_tokenizer=stack1.tokenizer,
        clip_image_processor=stack1.image_processor,
        config=stack1.config,
        boxes_px=[],
        format_prompt=FORMAT_PROMPT,
        max_new_tokens=max_new_tokens,
        include_regions=False,  # global576 only
    )
    answer = str(pred).strip().splitlines()[0].strip().rstrip(".")
    print("\nAnswer:", answer)
    print(f"Prefix: {aux.get('num_global_tokens', 576)} global + 0 region (0 boxes)")
    return {
        "answer": answer,
        "num_regions": 0,
        "num_global_tokens": aux.get("num_global_tokens", 576),
        "num_region_tokens": 0,
    }

while True:
    image_input = input("\n[Stage1] Image path/URL/filename (or quit): ").strip()
    if image_input.lower() in {"quit", "q", "exit"}:
        break

    if image_input.startswith("http") or Path(image_input).is_file():
        image_path = image_input
    else:
        image_path = str(IMAGES_ROOT / image_input)

    question = input("Question (or quit): ").strip()
    if question.lower() in {"quit", "q", "exit"}:
        break
    if not question:
        continue

    ask_stage1(image_path, question)